# Alternative model families

We tested LightGBM, Extra Trees, and an RBF SVM for Smoking, plus LightGBM and Extra Trees for Gamma_GT. Reference: the user-identified three-seed V2 CSV scored **0.86380** on Kaggle; the screenshot gap to first is **0.00044**. Local results are separate from Kaggle scores.

This is a project-backed Python notebook. Retraining is disabled by default. It displays the exact implementations and saved experiment results.

In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display, Code
ROOT=Path.cwd()
assert (ROOT/'alternative_models.py').exists(), 'Run from the hackathon project root.'
for script in ['alternative_models.py','alternative_gamma.py','finalize_alternative_gamma.py']:
    display(Code(filename=str(ROOT/script),language='python'))

## Optional reproduction

Select the project Python environment. Selection uses three folds on development rows. Blend weights are chosen before a second fold assignment. Confirmation requires a combined-score gain above 0.0003 and at least two fold wins. Neither selected candidate cleared that gate. Gamma_GT finalization also requires a gain against the successful V2 OOF before writing a submission.

In [ ]:
RUN_RETRAIN=False
if RUN_RETRAIN:
    import subprocess
    commands=[['alternative_models.py','--phase','all'],['alternative_gamma.py'],['finalize_alternative_gamma.py']]
    for command in commands:
        subprocess.run([str(ROOT/'.venv/bin/python'),*command],cwd=ROOT,check=True)

In [ ]:
for folder in ['alternative_models','alternative_gamma']:
    directory=ROOT/'outputs'/folder
    print(folder)
    display(pd.read_csv(directory/'selection.csv'))
    for filename in ['frozen_config.json','confirmation.json','final_cv.json']:
        path=directory/filename
        if path.exists(): display(json.loads(path.read_text()))

## Decision

The 15% LightGBM Smoking blend initially improved AUC by **0.00054333**, but its confirmation gain shrank to **0.00021156**. The Gamma_GT LightGBM blend's confirmation gain was **0.00010201** in the combined score. Extra Trees and the SVM were weaker. No candidate passed confirmation, so there is no new submission from this round.

Development data have been reused across experiments. The second split is a stability check, not independent validation. Gamma_GT comparisons in this notebook use a separate calibrated clinical baseline, not the submitted V2 ensemble. The inherited V2 validation limitations remain documented in the reports.

In [ ]:
reference_path=ROOT/'outputs/v2/submission_3seed.csv'
assert hashlib.sha256(reference_path.read_bytes()).hexdigest()=='c49a7788060dd229e1c6072b463c7f90233627f9be63cb4ad59bea4b8d0bd95a'
reference=pd.read_csv(reference_path)
sample=pd.read_csv(ROOT/'data/sample_submission.csv')
assert len(reference)==6399 and reference.ID.equals(sample.ID)
assert reference.columns.tolist()==['ID','Smoking','Gamma_GT']
assert np.isfinite(reference[['Smoking','Gamma_GT']]).all().all()
assert reference.Smoking.between(0,1).all() and reference.Gamma_GT.between(1,1000).all()
print('Confirmed Kaggle submission is intact and valid.')